In [1]:
# == Beyond Reconstruction - GLAS Gland Segmentation Dataset ==

!pip install -q torch torchvision matplotlib numpy scipy scikit-image opencv-python-headless tqdm scikit-learn statsmodels

import os, math, hashlib, copy
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt, gaussian_filter
from scipy.spatial import KDTree
from skimage import measure
from skimage.segmentation import find_boundaries
from scipy.stats import wilcoxon, spearmanr, t as t_dist
import torch, torch.nn as nn
from torch.optim import Adam
from PIL import Image
import cv2
from itertools import combinations
from statsmodels.stats.multitest import multipletests
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

SAVE_PLOTS          = True
COMPOSITE_FIDELITY  = True
BOOTSTRAP_CKA_CI    = False
COMPUTE_ICC         = False
SUMMARY_FIGURE      = True
DICE_EQUIV_BOUND     = 0.02
BOOTSTRAP_MASTER_SEED = 12345

DATASET_ROOTS = {'GLAS': '/kaggle/input/datasets/sani84/glasmiccai2015-gland-segmentation/Warwick_QU_Dataset'}
cv2.setNumThreads(0)
MAX_SAMPLES = 20         

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    if device.type == 'cuda':
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    BASE_SEEDS = [42, 123]            
    IMG_SIZE = (256, 256)             

    # DATA LOADING 
    def load_glas_dataset(root):
        """Load masks """
        data = []
        if not os.path.isdir(root): return []
        for f in sorted(os.listdir(root)):
            if f.endswith('.bmp') and not f.endswith('_anno.bmp'):
                base = f[:-4]
                anno_file = base + '_anno.bmp'
                anno_path = os.path.join(root, anno_file)
                if not os.path.exists(anno_path): continue
                img = Image.open(anno_path).convert('L')
                manual = cv2.resize(np.array(img).astype(np.float32), IMG_SIZE,
                                    interpolation=cv2.INTER_NEAREST)
                manual = (manual > 0).astype(np.float32)   
                if 0 < manual.sum() < manual.size:
                    data.append({'mask': manual, 'id': f"GLAS_{base}"})
        return data

    all_data = load_glas_dataset(DATASET_ROOTS['GLAS'])
    if len(all_data) == 0:
        raise RuntimeError("No valid non-empty masks found.")
    if MAX_SAMPLES is not None and MAX_SAMPLES > 0:
        all_data = all_data[:MAX_SAMPLES]
    print(f"GLAS (valid, non‑degenerate masks): {len(all_data)} samples")

    #  COORDINATE GRID 
    COORD_GRID_CACHE = {}
    def coord_grid(shape):
        if shape in COORD_GRID_CACHE: return COORD_GRID_CACHE[shape]
        h, w = shape
        x = np.linspace(-1, 1, w); y = np.linspace(-1, 1, h)
        xx, yy = np.meshgrid(x, y)
        grid = np.stack([xx, yy], axis=-1).reshape(-1, 2).astype(np.float32)
        COORD_GRID_CACHE[shape] = grid
        return grid

    shared_coords_np = coord_grid(IMG_SIZE)
    shared_coords_gpu = torch.from_numpy(shared_coords_np).to(device)

    # SDF HELPERS 
    def compute_sdf(mask):
        """Compute normalised SDF from binary mask. Assumes mask is not all 0 or all 1."""
        mask_bin = (mask > 0.5).astype(np.uint8)
        dist_in = distance_transform_edt(mask_bin)
        dist_out = distance_transform_edt(1 - mask_bin)
        sdf = dist_in - dist_out
        max_dist = max(dist_in.max(), dist_out.max())
        if max_dist <= 1e-8 or np.isinf(max_dist):
            return np.zeros_like(sdf, dtype=np.float32)
        sdf /= max_dist
        return sdf.astype(np.float32)

    # Pre-upload dataset tensors to GPU 
    gpu_data = {}
    print("Pre-uploading dataset tensors to GPU ...")
    for item in tqdm(all_data, desc="Uploading"):
        mask_np = item['mask']                     
        sdf_np = compute_sdf(mask_np)
        sdf_gpu = torch.from_numpy(sdf_np.flatten()).view(-1, 1).to(device)
        mask_gpu = torch.from_numpy(mask_np.flatten()).view(-1, 1).to(device)
        gpu_data[item['id']] = (sdf_gpu, mask_gpu)

    #  NETWORKS 
    class SineLayer(nn.Module):
        def __init__(self, in_f, out_f, is_first=False, omega0=30.0):
            super().__init__()
            self.omega0 = omega0
            self.linear = nn.Linear(in_f, out_f)
            with torch.no_grad():
                if is_first: self.linear.weight.uniform_(-1/in_f, 1/in_f)
                else: self.linear.weight.uniform_(-math.sqrt(6/in_f)/omega0, math.sqrt(6/in_f)/omega0)
        def forward(self, x): return torch.sin(self.omega0 * self.linear(x))

    class SIREN_SDF(nn.Module):
        def __init__(self, in_dim=2, hidden=128, layers=4, omega0=30.0):
            super().__init__()
            self.layers = nn.ModuleList()
            self.layers.append(SineLayer(in_dim, hidden, is_first=True, omega0=omega0))
            for _ in range(layers-2): self.layers.append(SineLayer(hidden, hidden, omega0=omega0))
            self.final = nn.Linear(hidden, 1)
        def forward(self, x, return_all_hidden=False):
            h = x; hidden_list = []
            for layer in self.layers:
                h = layer(h)
                if return_all_hidden: hidden_list.append(h)
            out = self.final(h)
            return (out, hidden_list) if return_all_hidden else out

    #  ACTIVATION HELPERS 
    @torch.inference_mode()
    def compute_hidden_activations_gpu(model, coords_gpu):
        model.eval()
        _, hidden_list = model(coords_gpu, return_all_hidden=True)
        return hidden_list

    def spectral_metrics_from_activations_gpu(H):
        H = H - H.mean(dim=0, keepdim=True)
        cov = H.T @ H / (H.shape[0] - 1)
        eigvals = torch.linalg.eigvalsh(cov)
        eigsum = eigvals.sum()
        if eigsum < 1e-12: return 0.0, 0.0, 0.0, 0.0
        pr = (eigsum**2) / (torch.sum(eigvals**2) + 1e-8)
        p = eigvals / (eigsum + 1e-8); p = p[p > 0]
        ent = -torch.sum(p * torch.log(p))
        eff_rank = torch.exp(ent).item()
        stable_rank = eigsum / (eigvals.max() + 1e-8)
        return pr.item(), ent.item(), eff_rank, stable_rank.item()

    def cka_similarity_gpu(H1, H2):
        H1 = H1 - H1.mean(dim=0); H2 = H2 - H2.mean(dim=0)
        cross = torch.linalg.norm(H1.T @ H2, 'fro')**2
        self1 = torch.linalg.norm(H1.T @ H1, 'fro')**2
        self2 = torch.linalg.norm(H2.T @ H2, 'fro')**2
        return (cross / (torch.sqrt(self1 * self2) + 1e-8)).item()

    def compute_pairwise_cka_from_acts(acts_dict):
        names = list(acts_dict.keys()); num_layers = len(acts_dict[names[0]])
        cka_results = {}
        for (n1, n2) in combinations(names, 2):
            layer_cka = [cka_similarity_gpu(acts_dict[n1][l], acts_dict[n2][l]) for l in range(num_layers)]
            cka_results[(n1, n2)] = layer_cka
        return cka_results

    def compute_intra_cka_from_acts(hidden_list):
        return [cka_similarity_gpu(hidden_list[l], hidden_list[l+1]) for l in range(len(hidden_list)-1)]

    def jacobian_sensitivity_gpu(model, coords_gpu):
        model.eval()
        coords = coords_gpu.clone().requires_grad_(True)
        out = model(coords)
        grad = torch.autograd.grad(out, coords, grad_outputs=torch.ones_like(out), create_graph=False)[0]
        return torch.norm(grad, p='fro', dim=1).mean().item()

    #  TRAINING 
    def train_sdf_gpu(model, coords_gpu, sdf_gpu,
                      epochs=2000, lr=1e-4, patience=800, eval_every=50,
                      return_losses=False, reg_weight=0.0, reg_type='eikonal'):
        model.train()
        opt = Adam(model.parameters(), lr=lr)
        mse = nn.MSELoss()
        best_loss = float('inf'); patience_cnt = 0
        best_state = copy.deepcopy(model.state_dict())   # fallback: initial state
        best_epoch = 0
        eval_losses, eval_epochs = [], []
        converged = True
        for ep in range(epochs):
            opt.zero_grad()
            out = model(coords_gpu)
            loss = mse(out, sdf_gpu)
            if reg_weight > 0:
                coords = coords_gpu.clone().requires_grad_(True)
                out_reg = model(coords)
                grad = torch.autograd.grad(out_reg, coords, grad_outputs=torch.ones_like(out_reg), create_graph=True)[0]
                reg = (torch.mean((torch.norm(grad, p=2, dim=1) - 1.0)**2) if reg_type=='eikonal'
                       else torch.mean(torch.norm(grad, p='fro', dim=1)))
                loss = loss + reg_weight * reg
            if torch.isnan(loss):
                converged = False
                break
            loss.backward(); opt.step()
            if ep % eval_every == 0 or ep == epochs - 1:
                model.eval()
                with torch.inference_mode(): eval_loss = mse(model(coords_gpu), sdf_gpu).item()
                model.train()
                if return_losses:
                    eval_losses.append(eval_loss)
                    eval_epochs.append(ep)
                if eval_loss < best_loss:
                    best_loss = eval_loss
                    patience_cnt = 0
                    best_epoch = ep
                    best_state = copy.deepcopy(model.state_dict())
                else:
                    patience_cnt += 1
                    if patience_cnt >= patience // eval_every:
                        break
        if not converged:
            best_loss = float('nan')
            best_epoch = 0
        model.load_state_dict(best_state)
        info = {'best_epoch': best_epoch, 'best_loss': best_loss, 'converged': converged}
        if return_losses:
            info['eval_losses'] = eval_losses; info['eval_epochs'] = eval_epochs
        return info

    #  RECONSTRUCTION METRICS 
    def dice(pred_bin, gt_bin):
        inter = np.sum(pred_bin * gt_bin)
        return 2*inter / (np.sum(pred_bin) + np.sum(gt_bin) + 1e-8)
    def hausdorff95(pred_bin, gt_bin):
        bound_p = find_boundaries(pred_bin, mode='thick'); bound_g = find_boundaries(gt_bin, mode='thick')
        pts_p, pts_g = np.argwhere(bound_p), np.argwhere(bound_g)
        if len(pts_p)==0 or len(pts_g)==0: return np.nan
        tree_p, tree_g = KDTree(pts_p), KDTree(pts_g)
        return max(np.percentile(tree_g.query(pts_p)[0], 95), np.percentile(tree_p.query(pts_g)[0], 95))
    def asd(pred_bin, gt_bin):
        bound_p = find_boundaries(pred_bin, mode='thick'); bound_g = find_boundaries(gt_bin, mode='thick')
        pts_p, pts_g = np.argwhere(bound_p), np.argwhere(bound_g)
        if len(pts_p)==0 or len(pts_g)==0: return np.nan
        tree_p, tree_g = KDTree(pts_p), KDTree(pts_g)
        return (np.mean(tree_g.query(pts_p)[0]) + np.mean(tree_p.query(pts_g)[0])) / 2.0
    def reconstruction_metrics(pred_field, gt_mask, field_type='sdf'):
        th = 0.0 if field_type=='sdf' else 0.5
        pred_bin = (pred_field > th).astype(np.float32)
        gt_bin = (gt_mask > 0.5).astype(np.float32)
        return {'Dice': dice(pred_bin, gt_bin), 'HD95': hausdorff95(pred_bin, gt_bin), 'ASD': asd(pred_bin, gt_bin)}

    #  GEOMETRY FIDELITY 
    def _boundary_mask_from_sdf(gt_sdf, eps=0.05, min_pixels=50):
        boundary = np.abs(gt_sdf) < eps
        if boundary.sum() < min_pixels:
            mask = gt_sdf <= 0
            dist = distance_transform_edt(mask) + distance_transform_edt(~mask)
            diag = np.sqrt(gt_sdf.shape[0]**2 + gt_sdf.shape[1]**2)
            boundary = dist < max(1.0, 0.02*diag)
        if boundary.sum() < 10: return None
        return boundary
    def _finite_difference_gradient(f): return np.gradient(f, axis=1), np.gradient(f, axis=0)
    def gradient_fidelity(pred_field, gt_sdf, eps=0.05):
        boundary = _boundary_mask_from_sdf(gt_sdf, eps)
        if boundary is None: return np.nan
        gp_x, gp_y = _finite_difference_gradient(pred_field)
        gg_x, gg_y = _finite_difference_gradient(gt_sdf)
        norm_p = np.sqrt(gp_x**2 + gp_y**2) + 1e-8; norm_g = np.sqrt(gg_x**2 + gg_y**2) + 1e-8
        cos = (gp_x*gg_x + gp_y*gg_y) / (norm_p * norm_g)
        return np.mean(np.arccos(np.clip(cos[boundary], -1, 1))) * 180/np.pi
    def _finite_difference_hessian(f):
        dfdx, dfdy = np.gradient(f, axis=1), np.gradient(f, axis=0)
        d2fdx2 = np.gradient(dfdx, axis=1); d2fdy2 = np.gradient(dfdy, axis=0)
        d2fdxdy = 0.5 * (np.gradient(dfdx, axis=0) + np.gradient(dfdy, axis=1))
        return d2fdx2, d2fdy2, d2fdxdy
    def curvature_fidelity(pred_field, gt_sdf, eps=0.05):
        boundary = _boundary_mask_from_sdf(gt_sdf, eps)
        if boundary is None: return np.nan
        pred_s = gaussian_filter(pred_field, sigma=0.5); g_s = gaussian_filter(gt_sdf, sigma=0.5)
        gxx_p, gyy_p, gxy_p = _finite_difference_hessian(pred_s)
        fx_p, fy_p = np.gradient(pred_s, axis=1), np.gradient(pred_s, axis=0)
        gxx, gyy, gxy = _finite_difference_hessian(g_s)
        fx, fy = np.gradient(g_s, axis=1), np.gradient(g_s, axis=0)
        G2_gt = fx**2 + fy**2 + 1e-8; G2_p = fx_p**2 + fy_p**2 + 1e-8
        mc_gt = (fx**2*gyy - 2*fx*fy*gxy + fy**2*gxx) / G2_gt**1.5
        mc_pred = (fx_p**2*gyy_p - 2*fx_p*fy_p*gxy_p + fy_p**2*gxx_p) / G2_p**1.5
        mae = np.mean(np.abs(mc_pred[boundary] - mc_gt[boundary]))
        norm_factor = np.mean(np.abs(mc_gt[boundary])) + 1e-3
        return mae / norm_factor
    def euler_characteristic_fidelity(pred_field, gt_bin, field_type='sdf'):
        gt_bool = gt_bin > 0.5
        if np.sum(gt_bool)==0: return np.nan
        euler_gt = measure.euler_number(gt_bool, connectivity=2)
        if field_type=='sdf':
            minv, maxv = np.nanmin(pred_field), np.nanmax(pred_field)
            if maxv - minv < 1e-8: return np.nan
            ths = np.linspace(minv, maxv, 50)
            sigma = max(0.15*(maxv-minv), 0.15)
            weights = np.exp(-(ths**2)/(2*sigma**2))
        else:
            ths = np.linspace(0.1, 0.9, 50)
            weights = np.exp(-((ths-0.5)**2)/(2*0.15**2))
        euler_pred = [measure.euler_number(pred_field>t, connectivity=2) for t in ths]
        errs = np.abs(np.array(euler_pred) - euler_gt)
        return np.average(errs, weights=weights)

    #  COMPOSITE RFI 
    def compute_fidelity_indices(df_wide):
        geo_cols = ['GradErr', 'CurvErr', 'EulerCharErr']
        latent_cols = ['EffectiveRank', 'SpectralEntropy', 'StableRank', 'ParticipationRatio', 'CoordSens']
        siren_mask = df_wide['model'].str.startswith('SIREN')
        geo_z = np.full((len(df_wide), len(geo_cols)), np.nan)
        for j, col in enumerate(geo_cols):
            valid_idx = siren_mask & df_wide[col].notna()
            if valid_idx.sum() < 2: continue
            ref_vals = df_wide.loc[valid_idx, col].values.astype(float)
            mean = np.nanmean(ref_vals); std = np.nanstd(ref_vals)
            if std < 1e-12: continue
            all_vals = df_wide[col].values.astype(float)
            geo_z[:, j] = (all_vals - mean) / std
        geo_index = np.full(len(df_wide), np.nan)
        valid_geo = ~np.isnan(geo_z)
        if valid_geo.any(): geo_index[valid_geo.any(axis=1)] = np.nanmean(geo_z, axis=1)[valid_geo.any(axis=1)]
        latent_data = df_wide[latent_cols].values.astype(float)
        latent_z = np.zeros_like(latent_data)
        for j in range(latent_data.shape[1]):
            col = latent_data[:, j]
            mean = np.nanmean(col); std = np.nanstd(col)
            if std > 1e-12: latent_z[:, j] = (col - mean) / std
            else: latent_z[:, j] = np.nan
        latent_index = -np.nanmean(latent_z, axis=1)
        rfi = (geo_index + latent_index) / 2.0   # Lower RFI = better fidelity
        df_out = df_wide.copy()
        df_out['GeoIndex'] = geo_index; df_out['LatentIndex'] = latent_index; df_out['RFI'] = rfi
        return df_out

    #  MODEL CONFIGS 
    MODEL_CONFIGS = {
        'SIREN-128':          {'type':'siren','hidden':128,'layers':4, 'omega':30.0, 'reg_weight':0.0, 'reg_type':'eikonal','lr':5e-5},
        'SIREN-128-Eikonal':  {'type':'siren','hidden':128,'layers':4, 'omega':30.0, 'reg_weight':1e-4,'reg_type':'eikonal','lr':5e-5},
        'SIREN-128-LowOmega': {'type':'siren','hidden':128,'layers':4, 'omega':15.0, 'reg_weight':0.0, 'reg_type':'eikonal','lr':5e-5},
        'SIREN-128-Jacobian': {'type':'siren','hidden':128,'layers':4, 'omega':30.0, 'reg_weight':1e-4,'reg_type':'jacobian','lr':5e-5},
    }

    #  CORE EXPERIMENT 
    def run_one_image_seed(item_id, seed, coords_gpu, sdf_gpu, mask_gpu, model_name, config):
        np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        model = SIREN_SDF(in_dim=2, hidden=config['hidden'], layers=config['layers'], omega0=config.get('omega',30.0)).to(device)
        try:
            info = train_sdf_gpu(model, coords_gpu, sdf_gpu, epochs=2000, lr=config['lr'], patience=800, eval_every=50,
                                 return_losses=True, reg_weight=config.get('reg_weight',0.0), reg_type=config.get('reg_type','eikonal'))
        except Exception as e:
            print(f"Warning: Unexpected error for {item_id} seed {seed} model {model_name}: {e}. Returning NaN.")
            info = {'converged': False, 'best_loss': float('nan'), 'best_epoch': 0}

        if not info.get('converged', False):
            return {'image': item_id, 'seed': seed, 'model': model_name,
                    'recon': {'Dice': np.nan, 'HD95': np.nan, 'ASD': np.nan},
                    'geo': {'GradErr': np.nan, 'CurvErr': np.nan, 'EulerCharErr': np.nan},
                    'latent': {'EffectiveRank': np.nan, 'SpectralEntropy': np.nan, 'StableRank': np.nan,
                               'ParticipationRatio': np.nan, 'CoordSens': np.nan},
                    'layer_metrics': [], 'intra_cka': [], 'convergence': info,
                    'hidden_activations': None}

        model.eval()
        with torch.inference_mode(): pred_field = model(coords_gpu).cpu().numpy().reshape(IMG_SIZE)
        sdf_gt_np = sdf_gpu.cpu().numpy().reshape(IMG_SIZE); mask_np = mask_gpu.cpu().numpy().reshape(IMG_SIZE)
        recon = reconstruction_metrics(pred_field, mask_np, 'sdf')
        hidden_list = compute_hidden_activations_gpu(model, coords_gpu)
        layer_metrics = [spectral_metrics_from_activations_gpu(H) for H in hidden_list]
        intra_cka = compute_intra_cka_from_acts(hidden_list)   # stored but not further analysed
        jac_sens = jacobian_sensitivity_gpu(model, coords_gpu)
        curv_err = curvature_fidelity(pred_field, sdf_gt_np); grad_err = gradient_fidelity(pred_field, sdf_gt_np)
        euler = euler_characteristic_fidelity(pred_field, mask_np, 'sdf')
        res_dict = {
            'image': item_id, 'seed': seed, 'model': model_name,
            'recon': recon,
            'geo': {'GradErr': grad_err, 'CurvErr': curv_err, 'EulerCharErr': euler},
            'latent': {'EffectiveRank': layer_metrics[-1][2], 'SpectralEntropy': layer_metrics[-1][1],
                       'StableRank': layer_metrics[-1][3], 'ParticipationRatio': layer_metrics[-1][0],
                       'CoordSens': jac_sens},
            'layer_metrics': layer_metrics, 'intra_cka': intra_cka, 'convergence': info,
            'hidden_activations': hidden_list,
        }
        del model, hidden_list; torch.cuda.empty_cache()
        return res_dict

    #  MAIN LOOP 
    all_results = []; cka_data_all = []
    for idx, item in enumerate(tqdm(all_data, desc="Images")):
        img_id = item['id']; sdf_gpu, mask_gpu = gpu_data[img_id]
        for seed in BASE_SEEDS:
            per_seed_acts = {}
            for model_name, config in MODEL_CONFIGS.items():
                res = run_one_image_seed(img_id, seed, shared_coords_gpu, sdf_gpu, mask_gpu, model_name, config)
                if res['hidden_activations'] is not None: per_seed_acts[model_name] = res['hidden_activations']
                all_results.append({
                    'image': img_id, 'seed': seed, 'model': model_name,
                    'recon': res['recon'], 'geo': res['geo'], 'latent': res['latent'],
                    'layer_metrics': res['layer_metrics'], 'intra_cka': res['intra_cka'], 'conv': res['convergence']
                })
            if len(per_seed_acts) >= 2: cka_res = compute_pairwise_cka_from_acts(per_seed_acts); cka_data_all.append((img_id, cka_res))
            del per_seed_acts; torch.cuda.empty_cache()

    #  DATA AGGREGATION 
    df = pd.DataFrame(all_results)
    for m in ['Dice','HD95','ASD']: df[m] = df['recon'].apply(lambda x: x[m])
    geo_metrics = ['GradErr','CurvErr','EulerCharErr']
    for m in geo_metrics: df[m] = df['geo'].apply(lambda x: x.get(m, np.nan))
    latent_metrics = ['EffectiveRank','SpectralEntropy','StableRank','ParticipationRatio','CoordSens']
    for m in latent_metrics: df[m] = df['latent'].apply(lambda x: x.get(m, np.nan))
    agg_df = df.drop(columns=['recon','geo','latent','layer_metrics','intra_cka','conv','seed']).groupby(['image','model']).mean().reset_index()
    if COMPOSITE_FIDELITY: agg_df = compute_fidelity_indices(agg_df); print("Composite fidelity indices computed.")

    #  STATISTICAL HELPERS 
    def _seed_for_metric(metric, model='', m1='', m2=''):
        base = f"{BOOTSTRAP_MASTER_SEED}_{metric}_{model}_{m1}_{m2}"
        return abs(int(hashlib.md5(base.encode()).hexdigest(), 16)) % (2**31 - 1)

    def bootstrap_ci_mean(vals, n_boot=500, metric='', model=''):
        vals = np.array(vals, dtype=float); vals = vals[np.isfinite(vals)]
        n_valid = len(vals)
        if n_valid < 3: return [np.nan, np.nan], n_valid
        rng = np.random.RandomState(_seed_for_metric(metric, model))
        stats = [np.mean(vals[rng.choice(n_valid, n_valid, replace=True)]) for _ in range(n_boot)]
        return np.percentile(stats, [2.5, 97.5]), n_valid

    def bootstrap_ci_diff(diffs, n_boot=500, metric='', m1='', m2=''):
        diffs = np.array(diffs, dtype=float); diffs = diffs[np.isfinite(diffs)]
        n = len(diffs)
        if n < 3: return [np.nan, np.nan]
        rng = np.random.RandomState(_seed_for_metric(metric, m1=m1, m2=m2))
        stats = [np.mean(diffs[rng.choice(n, n, replace=True)]) for _ in range(n_boot)]
        return np.percentile(stats, [2.5, 97.5])

    def paired_diff_table(df_wide, m1, m2, metric):
        try: t = df_wide.pivot(index='image', columns='model', values=metric)[[m1,m2]].dropna()
        except KeyError: return None
        if len(t) < 3: return None
        return t[m1].values - t[m2].values

    def safe_wilcoxon_paired(diffs):
        if diffs is None or len(diffs) < 3: return np.nan, np.nan
        try: res = wilcoxon(diffs, method='auto'); return res.pvalue, res.zstatistic
        except: return np.nan, np.nan

    def effect_size_r(diffs):
        _, z = safe_wilcoxon_paired(diffs)
        if z is None or np.isnan(z): return np.nan
        return z / np.sqrt(len(diffs))

    def tost_equivalence(diffs, bound, alpha=0.05):
        diffs = np.array(diffs, dtype=float); diffs = diffs[np.isfinite(diffs)]
        n = len(diffs)
        if n < 3: return False, None, None, None
        mean_diff = np.mean(diffs); sd = np.std(diffs, ddof=1); se = sd / np.sqrt(n)
        if se < 1e-12: se = 1e-12   # treat zero variance as trivially equivalent
        tcrit = t_dist.ppf(0.95, df=n-1); ci90 = (mean_diff - tcrit*se, mean_diff + tcrit*se)
        t_lower = (mean_diff + bound) / se; p_lower = 1 - t_dist.cdf(t_lower, df=n-1)
        t_upper = (mean_diff - bound) / se; p_upper = t_dist.cdf(t_upper, df=n-1)
        p_tost = max(p_lower, p_upper)
        return p_tost < alpha, mean_diff, p_tost, (mean_diff, sd, n, tcrit, ci90)

    #  FULL STATISTICAL ANALYSIS 
    models = list(MODEL_CONFIGS.keys())
    metric_list = ['Dice','HD95','ASD'] + geo_metrics + latent_metrics
    if COMPOSITE_FIDELITY: metric_list += ['GeoIndex','LatentIndex','RFI']

    print("\n=== MEAN ± 95% CI ===")
    for met in metric_list:
        print(f"\n{met}:")
        for m in models:
            vals = agg_df[agg_df['model']==m][met].dropna().values
            if len(vals) == 0: continue
            ci, n = bootstrap_ci_mean(vals, metric=met, model=m)
            print(f"  {m}: {np.mean(vals):.3f} [{ci[0]:.3f}, {ci[1]:.3f}], n={n}")

    pvals_all = []
    pairs = list(combinations(models,2))
    for met in metric_list:
        for m1,m2 in pairs:
            if met in agg_df.columns:
                diffs = paired_diff_table(agg_df, m1, m2, met)
                p, _ = safe_wilcoxon_paired(diffs)
                pvals_all.append((met, m1, m2, p))
    valid_p = [x for x in pvals_all if not np.isnan(x[3])]
    if len(valid_p) > 0:
        reject, p_corr, _, _ = multipletests([p for _,_,_,p in valid_p], method='holm')
        print("\n=== WILCOXON SIGNED-RANK (Holm-adjusted p-values) ===")
        for (met,m1,m2,p), pc, r in zip(valid_p, p_corr, reject):
            print(f"{met} {m1} vs {m2}: p={p:.4f}, p_holm={pc:.4f}, reject={r}")
    else:
        print("\nNo valid pairs for Wilcoxon tests.")

    #  SPEARMAN CORRELATION 
    print("\n=== SPEARMAN CORRELATION WITH DICE (per model) ===")
    corr_pvals = []; corr_info = []
    for met in metric_list:
        if met == 'Dice': continue
        for m in models:
            sub = agg_df[agg_df['model']==m].dropna(subset=['Dice', met])
            if len(sub) > 5:
                r, p = spearmanr(sub['Dice'], sub[met])
                if not np.isnan(p):          # only add valid p-values
                    corr_pvals.append(p)
                    corr_info.append((m, met, r, p))
    if corr_pvals:
        reject_corr, p_corr_corr, _, _ = multipletests(corr_pvals, method='holm')
        for (m, met, r, p), pc, rej in zip(corr_info, p_corr_corr, reject_corr):
            print(f"{m} Dice vs {met}: r={r:.3f}, p={p:.4f}, p_holm={pc:.4f}, {'*' if rej else 'n.s.'}")
    else:
        print("No valid Spearman correlations for correction.")

    print(f"\n=== DICE EQUIVALENCE (TOST, bound=±{DICE_EQUIV_BOUND}) ===")
    tost_results = []
    for m1,m2 in pairs:
        diffs = paired_diff_table(agg_df, m1, m2, 'Dice')
        if diffs is not None and len(diffs) >= 3:
            is_equiv, mean_diff, p_tost, stats = tost_equivalence(diffs, DICE_EQUIV_BOUND)
            tost_results.append((m1, m2, is_equiv, mean_diff, p_tost, stats))
    tost_pvals = [p for (_, _, _, _, p, _) in tost_results if p is not None]
    equiv_pairs = []
    if tost_pvals:
        reject_tost, p_corr_tost, _, _ = multipletests(tost_pvals, method='holm')
        idx = 0
        for i, (m1, m2, _, mean_diff, raw_p, stats) in enumerate(tost_results):
            corr_p = p_corr_tost[idx] if raw_p is not None else np.nan
            is_equiv_corr = corr_p < 0.05
            if stats is not None:
                mean_diff, sd, n, tcrit, ci90 = stats
                print(f"  {m1} vs {m2}: ΔDice = {mean_diff:.4f}, 90% CI [{ci90[0]:.4f}, {ci90[1]:.4f}], "
                      f"TOST p={raw_p:.4f} (Holm-adj p={corr_p:.4f}), {'EQUIVALENT' if is_equiv_corr else 'NOT EQUIVALENT'}")
            if is_equiv_corr: equiv_pairs.append((m1, m2))
            idx += 1
    else:
        print("  (insufficient data for TOST)")

    if equiv_pairs:
        print("\n=== FOCUS: Equivalent Dice pairs (SIREN variants only) ===")
        focus_metrics = ['RFI','GeoIndex','LatentIndex','GradErr','CurvErr']
        focus_pvals = []; focus_info = []
        for m1,m2 in equiv_pairs:
            diffs_dice = paired_diff_table(agg_df, m1, m2, 'Dice')
            diffs_hd95 = paired_diff_table(agg_df, m1, m2, 'HD95')
            diffs_asd  = paired_diff_table(agg_df, m1, m2, 'ASD')
            print(f"\n{m1} vs {m2}")
            if diffs_dice is not None: print(f"  ΔDice = {np.mean(diffs_dice):.4f} (p={safe_wilcoxon_paired(diffs_dice)[0]:.4f})")
            if diffs_hd95 is not None: print(f"  ΔHD95 = {np.mean(diffs_hd95):.3f} (p={safe_wilcoxon_paired(diffs_hd95)[0]:.4f})")
            if diffs_asd is not None: print(f"  ΔASD  = {np.mean(diffs_asd):.4f} (p={safe_wilcoxon_paired(diffs_asd)[0]:.4f})")
            print("  Fidelity metrics:")
            for met in focus_metrics:
                if met in agg_df.columns:
                    diffs = paired_diff_table(agg_df, m1, m2, met)
                    if diffs is not None:
                        p, _ = safe_wilcoxon_paired(diffs); r_rb = effect_size_r(diffs)
                        ci_diff = bootstrap_ci_diff(diffs, metric=met, m1=m1, m2=m2)
                        focus_pvals.append(p); focus_info.append((m1, m2, met, r_rb, p, ci_diff))
                        print(f"    {met}: Δ = {np.mean(diffs):.4f} 95% CI [{ci_diff[0]:.4f}, {ci_diff[1]:.4f}], "
                              f"r = {r_rb:.3f}, raw p={p:.4f}")
        if focus_pvals:
            valid_focus = ~np.isnan(focus_pvals)
            clean_focus_pvals = [p for p, v in zip(focus_pvals, valid_focus) if v]
            clean_focus_info = [x for x, v in zip(focus_info, valid_focus) if v]
            if clean_focus_pvals:
                reject_f, p_corr_f, _, _ = multipletests(clean_focus_pvals, method='holm')
                print("\n  Holm-adjusted p-values for fidelity metrics:")
                for (m1, m2, met, r_rb, raw_p, ci_diff), corr_p, rej in zip(clean_focus_info, p_corr_f, reject_f):
                    print(f"    {m1} vs {m2} {met}: r={r_rb:.3f}, p_holm={corr_p:.4f}, {'*' if rej else 'n.s.'}")
            else:
                print("\n  No valid p-values for fidelity metrics after filtering NaN.")
        else:
            print("\n  No focus p-values to correct.")
    else:
        print("\nNo equivalent Dice pairs found.")

    if COMPOSITE_FIDELITY and equiv_pairs:
        print("\n=== RFI SENSITIVITY TO WEIGHTING ===")
        weights_pvals = []; weights_info = []
        for w_geo in [0.3, 0.5, 0.7]:
            w_lat = 1.0 - w_geo
            geo_cols = ['GradErr','CurvErr','EulerCharErr']
            latent_cols = ['EffectiveRank','SpectralEntropy','StableRank','ParticipationRatio','CoordSens']
            siren_mask = agg_df['model'].str.startswith('SIREN')
            geo_z = np.full((len(agg_df), len(geo_cols)), np.nan)
            for j, col in enumerate(geo_cols):
                valid_idx = siren_mask & agg_df[col].notna()
                if valid_idx.sum() < 2: continue
                ref_vals = agg_df.loc[valid_idx, col].values.astype(float)
                mean = np.nanmean(ref_vals); std = np.nanstd(ref_vals)
                if std < 1e-12: continue
                all_vals = agg_df[col].values.astype(float)
                geo_z[:, j] = (all_vals - mean) / std
            geo_idx = np.nanmean(geo_z, axis=1)
            latent_data = agg_df[latent_cols].values.astype(float)
            latent_z = np.zeros_like(latent_data)
            for j in range(latent_data.shape[1]):
                col = latent_data[:, j]
                mean = np.nanmean(col); std = np.nanstd(col)
                if std > 1e-12: latent_z[:, j] = (col - mean) / std
            latent_idx = -np.nanmean(latent_z, axis=1)
            rfi_w = w_geo*geo_idx + w_lat*latent_idx
            tmp = agg_df.copy(); tmp['RFI_w'] = rfi_w
            print(f"  Weights (Geo={w_geo:.1f}, Lat={w_lat:.1f}):")
            for m1,m2 in equiv_pairs:
                diffs = paired_diff_table(tmp, m1, m2, 'RFI_w')
                if diffs is not None:
                    p, _ = safe_wilcoxon_paired(diffs); r_rb = effect_size_r(diffs)
                    ci_diff = bootstrap_ci_diff(diffs, metric=f'RFI_w{w_geo}', m1=m1, m2=m2)
                    weights_pvals.append(p); weights_info.append((w_geo, m1, m2, r_rb, p, ci_diff))
                    print(f"    {m1} vs {m2}: ΔRFI_w = {np.mean(diffs):.4f} 95% CI [{ci_diff[0]:.4f}, {ci_diff[1]:.4f}], "
                          f"r={r_rb:.3f}, raw p={p:.4f}")
        if weights_pvals:
            valid_weights = ~np.isnan(weights_pvals)
            clean_weights_pvals = [p for p, v in zip(weights_pvals, valid_weights) if v]
            clean_weights_info = [x for x, v in zip(weights_info, valid_weights) if v]
            if clean_weights_pvals:
                reject_w, p_corr_w, _, _ = multipletests(clean_weights_pvals, method='holm')
                print("\n  Holm-adjusted p-values across all weights:")
                for (w_geo, m1, m2, r_rb, raw_p, ci_diff), corr_p, rej in zip(clean_weights_info, p_corr_w, reject_w):
                    print(f"    w_geo={w_geo:.1f} {m1} vs {m2}: r={r_rb:.3f}, p_holm={corr_p:.4f}, {'*' if rej else 'n.s.'}")
            else:
                print("\n  No valid p-values for sensitivity analysis.")
        else:
            print("\n  No sensitivity p-values to correct.")
    else:
        if COMPOSITE_FIDELITY and not equiv_pairs:
            print("\nNo equivalent Dice pairs for RFI sensitivity analysis.")

    print("\n=== LATENT METRIC CORRELATIONS ===")
    latent_df = agg_df[['EffectiveRank','SpectralEntropy','StableRank','ParticipationRatio','CoordSens']].dropna()
    corr = latent_df.corr(); print(corr.round(3))

    if SAVE_PLOTS:
        try:
            conv_models = [m for m in models if m.startswith('SIREN')][:4]
            if conv_models:
                first_res = df[(df['image']==all_data[0]['id']) & (df['seed']==BASE_SEEDS[0])]
                if len(first_res) > 0:
                    n_plots = len(conv_models)
                    fig, axes = plt.subplots(1, n_plots, figsize=(n_plots*4, 4))
                    if n_plots == 1: axes = [axes]
                    for ax, m in zip(axes, conv_models):
                        row = first_res[first_res['model']==m].iloc[0]
                        conv_dict = row['conv']
                        if 'eval_losses' in conv_dict and 'eval_epochs' in conv_dict:
                            ax.plot(conv_dict['eval_epochs'], conv_dict['eval_losses'])
                        ax.set_title(m); ax.set_xlabel('epoch'); ax.set_ylabel('eval loss'); ax.grid(True)
                    plt.tight_layout(); plt.savefig('convergence_selected.png', dpi=150); plt.close()
            if COMPOSITE_FIDELITY and SUMMARY_FIGURE and equiv_pairs:
                fig, ax = plt.subplots(1,2,figsize=(12,5))
                for m1,m2 in equiv_pairs:
                    dd = paired_diff_table(agg_df, m1, m2, 'Dice'); dr = paired_diff_table(agg_df, m1, m2, 'RFI')
                    if dd is not None and dr is not None: ax[0].scatter(np.mean(dd), np.mean(dr), label=f'{m1} vs {m2}')
                ax[0].axhline(0, color='gray', linestyle='--'); ax[0].set_xlabel('Δ Dice'); ax[0].set_ylabel('Δ RFI')
                ax[0].set_title('Equivalent Dice -> distinct fidelity'); ax[0].legend()
                if equiv_pairs:
                    m1,m2 = equiv_pairs[0]
                    comps = ['GeoIndex','LatentIndex']
                    vals1 = [agg_df[agg_df['model']==m1][c].mean() for c in comps]
                    vals2 = [agg_df[agg_df['model']==m2][c].mean() for c in comps]
                    x = np.arange(len(comps))
                    ax[1].bar(x-0.2, vals1, 0.4, label=m1); ax[1].bar(x+0.2, vals2, 0.4, label=m2)
                    ax[1].set_xticks(x); ax[1].set_xticklabels(comps); ax[1].set_ylabel('Fidelity index (z‑score)'); ax[1].legend()
                plt.tight_layout(); plt.savefig('summary_figure.png', dpi=150); plt.close()
        except Exception as e: print(f"Warning: Could not save plots ({e})")

    print("\nAll experiments completed successfully.")

if __name__ == '__main__':
    main()

Device: cuda
GLAS (valid, non‑degenerate masks): 20 samples
Pre-uploading dataset tensors to GPU ...


Images: 100%|██████████| 20/20 [1:57:37<00:00, 352.87s/it]


Composite fidelity indices computed.

=== MEAN ± 95% CI ===

Dice:
  SIREN-128: 0.995 [0.994, 0.996], n=20
  SIREN-128-Eikonal: 0.995 [0.994, 0.996], n=20
  SIREN-128-LowOmega: 0.986 [0.983, 0.989], n=20
  SIREN-128-Jacobian: 0.995 [0.994, 0.996], n=20

HD95:
  SIREN-128: 1.853 [1.000, 3.560], n=20
  SIREN-128-Eikonal: 2.150 [1.000, 4.411], n=20
  SIREN-128-LowOmega: 5.427 [1.544, 10.900], n=20
  SIREN-128-Jacobian: 2.082 [1.000, 4.246], n=20

ASD:
  SIREN-128: 0.151 [0.086, 0.264], n=20
  SIREN-128-Eikonal: 0.184 [0.094, 0.348], n=20
  SIREN-128-LowOmega: 0.591 [0.270, 1.002], n=20
  SIREN-128-Jacobian: 0.177 [0.087, 0.335], n=20

GradErr:
  SIREN-128: 10.583 [10.070, 11.138], n=20
  SIREN-128-Eikonal: 10.577 [10.001, 11.226], n=20
  SIREN-128-LowOmega: 12.449 [11.522, 13.273], n=20
  SIREN-128-Jacobian: 10.515 [9.958, 11.034], n=20

CurvErr:
  SIREN-128: 1.010 [0.991, 1.031], n=20
  SIREN-128-Eikonal: 1.007 [0.988, 1.028], n=20
  SIREN-128-LowOmega: 1.042 [1.022, 1.058], n=20
  SIREN

## Results and Hypothesis Decision

### Quantitative Findings

Four SIREN variants were trained to regress signed distance functions (SDFs) for 20 GLAS gland segmentation masks. Reconstruction performance was consistently high across all models, as measured by the Dice coefficient:

| Model              | Mean Dice [95% CI]   |
| ------------------ | -------------------- |
| SIREN-128          | 0.995 [0.994, 0.996] |
| SIREN-128-Eikonal  | 0.995 [0.994, 0.996] |
| SIREN-128-LowOmega | 0.986 [0.983, 0.989] |
| SIREN-128-Jacobian | 0.995 [0.994, 0.996] |

Pairwise two one-sided tests (TOST; equivalence bound ±0.02 Dice, Holm-corrected) indicated that all six model pairs were statistically equivalent with respect to Dice reconstruction performance (adjusted *p* < 0.0001 for every comparison). Thus, according to the Dice coefficient, reconstruction accuracy was statistically equivalent across all four SIREN variants.

In contrast, the representational fidelity analysis revealed substantial differences between the LowOmega (ω = 15) model and the three ω = 30 variants. The largest differences were observed in measures of latent representational richness:

| Metric              | ω = 30 Models (Range) | LowOmega (ω = 15) | Difference  |
| ------------------- | --------------------- | ----------------- | ----------- |
| Effective rank      | 74.08-74.44           | 43.54             | −41%        |
| Spectral entropy    | 4.305-4.310           | 3.771             | −12%        |
| Participation ratio | 51.41-51.73           | 28.46             | −45%        |
| Composite RFI       | −0.322 to −0.296      | +0.920            | ΔRFI ≈ 1.22 |

The composite Representational Fidelity Index (RFI) integrates standardized geometric and latent representation measures, with lower values indicating better representational fidelity. The three ω = 30 models consistently exhibited lower RFI values than the LowOmega model, with non-overlapping 95% confidence intervals. These findings remained consistent across sensitivity analyses in which the relative weighting of the geometric and latent components was varied from 0.3/0.7 to 0.7/0.3, indicating that the observed differences were not dependent on the specific weighting scheme.

Geometric fidelity metrics, including gradient error, curvature error, and Euler characteristic error, also consistently favoured the ω = 30 models, although the magnitude of these differences was smaller than that observed for the latent representation metrics.

For several representational fidelity measures, every paired comparison between the LowOmega model and the ω = 30 models exhibited identical sign and magnitude across all evaluated images. Under these deterministic paired differences, the Wilcoxon signed-rank statistic could not be computed because no ranking variability existed. This reflects complete consistency of the observed differences within the present dataset rather than variability that can be assessed using a rank-based statistical test.

### Hypothesis Decision

The central hypothesis of this study was:

> *Continuous neural representations that achieve similar reconstruction performance may preserve structurally and geometrically meaningful information with different levels of representational fidelity.*

The experimental results support this hypothesis. Although the LowOmega (ω = 15) model achieved Dice reconstruction performance statistically equivalent to that of the three ω = 30 models, it consistently exhibited substantially lower representational richness, characterized by reduced effective rank, spectral entropy, and participation ratio, together with a higher composite Representational Fidelity Index. These differences were observed despite equivalent reconstruction accuracy.

As a result, reconstruction quality alone did not distinguish the substantial differences in representational fidelity observed among the learned continuous neural representations. These findings indicate that reconstruction fidelity and representational fidelity capture complementary properties of implicit neural representations. Incorporating direct measures of representational fidelity alongside conventional reconstruction metrics may therefore provide a more comprehensive evaluation of continuous neural representations than reconstruction accuracy alone.

The present conclusions are limited to the evaluated SIREN architectures and the GLAS gland segmentation task. Validation on additional continuous neural representation architectures, anatomical structures, and medical imaging datasets will be necessary to establish the broader generality of these findings.
